In [1]:
import os
import io
import time
import requests
import pandas as pd
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# 1. 초기 설정
url = "https://poomgo.com/blog/e-commerce/2025-japan-market-analysis-k-beauty-trends"
PROJECT_NAME = "Poomgo_2025_Analysis"
IMAGE_DIR = os.path.join(PROJECT_NAME, "images")
os.makedirs(IMAGE_DIR, exist_ok=True)

options = Options()
# options.add_argument("--headless") # 필요 시 주석 해제
driver = webdriver.Chrome(options=options)

def flatten_cols(df):
    """지저분한 멀티 헤더 정리 함수"""
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = ['_'.join([str(c) for c in col if 'Unnamed' not in str(c)]).strip() for col in df.columns.values]
    return df

try:
    print(f"🔄 페이지 접속 및 동적 로딩 대기 중...")
    driver.get(url)

    # 2. 본문 구역 대기 (부분 일치 선택자 활용)
    wait = WebDriverWait(driver, 15)
    target_selector = 'div[class*="PostLayout_content_section"]'
    wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, target_selector)))
    
    # 스크롤을 내려 모든 요소 로드 유도
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight/2);")
    time.sleep(2)

    # 3. 데이터 파싱 시작
    soup = BeautifulSoup(driver.page_source, 'html.parser')
    content_area = soup.select_one(target_selector)

    if content_area:
        print("✅ 타겟 구역을 찾았습니다. 데이터 정규화 추출을 시작합니다.")
        
        table_md_results = []
        excel_count = 0

        # [A] 표(Table) 추출 및 엑셀 저장 (정규표 방식)
        tables = content_area.find_all('table')
        for t_idx, table in enumerate(tables, 1):
            try:
                df = pd.read_html(io.StringIO(str(table)))[0]
                df = flatten_cols(df)
                
                # 엑셀 파일 저장 (정규표 보존)
                excel_name = f"table_{t_idx}.xlsx"
                df.to_excel(os.path.join(PROJECT_NAME, excel_name), index=False)
                excel_count += 1
                
                # 프롬프트용 마크다운 저장
                table_md_results.append(f"\n[데이터 표 {t_idx}]\n{df.to_markdown(index=False)}\n")
                
                # 본문 텍스트 중복 방지를 위해 태그 제거
                table.decompose()
            except: pass

        # [B] 이미지 수집 및 저장
        img_tags = content_area.find_all('img')
        img_count = 0
        for i_idx, img in enumerate(img_tags, 1):
            src = img.get('src')
            if src:
                # 데이터 URL(base64)은 무시하고 실제 주소만 수집
                if not src.startswith('http'): 
                    src = requests.compat.urljoin(url, src)
                
                try:
                    res = requests.get(src, timeout=5)
                    if res.status_code == 200:
                        with open(os.path.join(IMAGE_DIR, f"img_{i_idx}.jpg"), 'wb') as f:
                            f.write(res.content)
                        img_count += 1
                except: pass

        # [C] 최종 텍스트 추출 (표가 제거된 깨끗한 상태)
        cleaned_text = content_area.get_text(separator='\n', strip=True)

        # 4. 프롬프트용 터미널 출력 (텍스트 파일 저장은 안 함)
        print("\n" + "🚀" * 15 + " [프롬프트 복사용 데이터] " + "🚀" * 15)
        print(f"SOURCE_URL: {url}")
        print("\n### [STRUCTURED TABLES]")
        print("".join(table_md_results) if table_md_results else "표 없음")
        print("\n### [CLEANED TEXT CONTENT]")
        print(cleaned_text)
        print("\n" + "🚀" * 40)

        print(f"\n✨ 수집 결과: 엑셀 {excel_count}개 저장, 이미지 {img_count}개 저장 완료")

    else:
        print("❌ 클래스 구역을 찾지 못했습니다.")

except Exception as e:
    print(f"❌ 오류 발생: {e}")

finally:
    driver.quit()
    print("\n🏁 모든 수집 작업이 종료되었습니다.")

🔄 페이지 접속 및 동적 로딩 대기 중...
✅ 타겟 구역을 찾았습니다. 데이터 정규화 추출을 시작합니다.

🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀 [프롬프트 복사용 데이터] 🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀
SOURCE_URL: https://poomgo.com/blog/e-commerce/2025-japan-market-analysis-k-beauty-trends

### [STRUCTURED TABLES]

[데이터 표 1]
| 0          | 1                                                                         |
|:-----------|:--------------------------------------------------------------------------|
| 성분 이름  | 효과                                                                      |
| 아젤리아산 | 염증 완화, 피부를 고르게 해 줌 (여드름)                                   |
| CICA(시카) | 피부장벽 강화, 진정효과                                                   |
| 레티놀     | 피부 재생 촉진, 주름 개선, 노화 방지                                      |
| PDRN       | 세포 재생과 회복, 피부 건강과 탄력 증진                                   |
| 글루타치온 | 강력한 향성화 성분으로 미백효과, 노화 방지, 식품으로 뷰티까지 영향을 끼침 |

[데이터 표 2]
| 0                                 | 1                                                                                               